# RSNA Knee — ensemble submission (probe #1)

**3-member ensemble.** Members, per-label blend method, and the TTA decision are all loaded from a **frozen, versioned recipe file** (`src/rsna_knee/ensemble_recipe_v1.yaml`, bundled in the `rsna-knee-code` dataset) — nothing is hardcoded in this notebook. A future recipe change is a new version of that file plus one path update, not a notebook edit.

Recipe v1 (2026-09-09/10, `scripts/build_ensemble.py`): greedy forward selection on OOF-vs-`labels_v2_final` macro-AUC (gold-58 used only as a tripwire, never for selection) chose:
- `exp006` — DINOv2 ViT-B/14 on `labels_v3_candidate`
- `exp003` — DINOv2 ViT-B/14 on `labels_v2_final`
- `exp005` — DINOv2 ViT-S/14 on `labels_v3_candidate`

"rank-mean" at single-study inference time has no population to rank against, so it's computed as an empirical-CDF lookup against each member's own stored OOF score distribution (`rank_reference.npz`, `infer_core.ensemble_blend`) rather than silently degraded to prob-mean.

**TTA (laterality-swap dual-pass): tested and REJECTED**, recorded in the recipe file itself (`tta.enabled: false`) — real inference passes on all 3 members x 5 folds measured OOF macro -0.0014 with TTA vs without (8/12 labels regressed).

**T4 x2 adaptation (2026-09-11 — the pinned accelerator is T4 x2, not H100):**
- **fp16, not bf16** (cell-2 `AUTOCAST_DTYPE`): T4 is sm_75 (Turing), no native bf16 tensor cores — this notebook previously relied on `infer_core`'s bf16 default (fine on H100, silently wrong here) and now passes fp16 explicitly at every call site.
- **Members sharded across cuda:0/cuda:1**, greedy-balanced by relative cost (ViT-B ≈ 3x ViT-S): `{exp006, exp005}` on one device, `exp003` alone on the other (`~4:3` load split, cell-2 `MEMBER_DEVICE`). Different members' forward passes run from separate Python threads (cell-5) so the two GPUs' CUDA kernels genuinely overlap — PyTorch releases the GIL during a CUDA call, so this isn't just concurrent bookkeeping. Falls back to one device automatically when only one GPU is visible (e.g. a normal HPC job).
- **Decode/inference overlap** (`infer_core.prefetch_decode`, cell-6): study N+1 decodes in a background thread pool while study N runs on the GPU(s), instead of the old fully-serial decode-then-infer-per-study loop.
- **Profiled H100 decode:model split is ~68:32** (job 25042) — meaningfully more model-bound than the efficiency notebook's ~82:18, which is exactly why dual-GPU sharding is worth it here (and not there, see `submit-efficiency.ipynb` cell-0). Projected T4 x2 wall-clock (3.5x model-time multiplier, decode unchanged — CPU-bound and already shown CPU-count-insensitive): **~62 min** for 1300 studies at rung 0, down from **~102 min projected on a single T4** — dual-GPU changes the bottleneck from model inference to decode. Both comfortably inside the 8h safety cap. Full numbers: `EXPERIMENTS.md` 2026-09-11.

**Degradation ladder** (CLAUDE.md §9 — a time-budget calibration picks ONE rung for the whole run, not a per-study drop, so scoring stays consistent across the submission):
1. All 3 members (the top rung, TTA never included since it's rejected)
2. {exp006, exp005} (drop exp003 — the older-labels ViT-B, most redundant given exp006 is its v3-label successor)
3. Smallest model only: exp005 (ViT-S, fastest)

A short calibration pass on the first studies (through the same decode-overlap pipeline as the main loop) measures actual per-study cost, projects total runtime per rung, and picks the highest rung that fits the budget with margin — then the wall-clock guard (per-study prior fallback) stays active underneath as defense-in-depth in case the projection was wrong.

**Runtime is a PROJECTION** until run on the actual pinned accelerator — hold this push until the 3-sample Kaggle test confirms the numbers above, per instruction.

Expects attached Kaggle Datasets: `rsna-knee-code`, `rsna-knee-exp006-weights`, `rsna-knee-exp003-weights`, `rsna-knee-exp005-weights`, plus the competition data. Internet OFF.

In [ ]:
import importlib.util
import os, sys, time, traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import torch

def _find_dir(root: str, name_contains: str, max_depth: int = 3) -> Path | None:
    root_path = Path(root)
    if not root_path.exists():
        return None
    
    # 1. Fast direct path checks for standard Kaggle layouts
    candidates = [
        root_path / name_contains,
        root_path / "competitions" / name_contains,
        root_path / "datasets" / name_contains,
        root_path / "datasets" / "hrdevelops" / name_contains,
    ]
    for c in candidates:
        if c.exists() and c.is_dir():
            return c

    # 2. Fast shallow BFS (up to 3 levels, skipping image series trees)
    queue = [(root_path, 0)]
    while queue:
        curr, depth = queue.pop(0)
        if depth > 0 and name_contains in curr.name:
            return curr
        if depth < max_depth:
            try:
                for entry in curr.iterdir():
                    if entry.is_dir():
                        if name_contains in entry.name:
                            return entry
                        if entry.name not in ("test_series", "train_series", "series", "images"):
                            queue.append((entry, depth + 1))
            except (PermissionError, OSError):
                pass
    return None

CODE_DIR = _find_dir("/kaggle/input", "rsna-knee-code")
TEST_DIR = _find_dir("/kaggle/input", "rsna-knee-abnormality-detection")
OUT_PATH = Path("/kaggle/working/submission.csv")

print("CODE_DIR:", CODE_DIR)
print("TEST_DIR:", TEST_DIR)
assert CODE_DIR is not None, "Attach rsna-knee-code"
assert TEST_DIR is not None, "Attach the competition data"

init_files = list(CODE_DIR.rglob("__init__.py"))
assert init_files, f"No __init__.py found under {CODE_DIR}"
pkg_dir = init_files[0].parent
spec = importlib.util.spec_from_file_location("rsna_knee", pkg_dir / "__init__.py", submodule_search_locations=[str(pkg_dir)])
rsna_knee = importlib.util.module_from_spec(spec)
sys.modules["rsna_knee"] = rsna_knee
spec.loader.exec_module(rsna_knee)

from rsna_knee.dataset import TARGET_COLUMNS, _patch_size_for_backbone
from rsna_knee.infer_core import (
    WallClockGuard, decode_one_for_pool, ensemble_blend, load_ensemble_recipe, nearest_patch_multiple,
    prefetch_decode, prior_row, run_study_inference, run_study_inference_tta,
)
from rsna_knee.model import SlotModel

# Frozen recipe (src/rsna_knee/ensemble_recipe_v1.yaml, bundled in the rsna-knee-code dataset) —
# members, per-label blend method, and the TTA decision all come from THIS file, not a hardcoded
# list here, so a future recipe version is a new file + one path change, not a notebook edit.
RECIPE_PATH = pkg_dir / "ensemble_recipe_v1.yaml"
assert RECIPE_PATH.exists(), f"Recipe file not found at {RECIPE_PATH} — check the rsna-knee-code dataset version"
loaded = load_ensemble_recipe(RECIPE_PATH)
MEMBERS = {name: {**spec} for name, spec in loaded["members"].items()}
RECIPE = loaded["recipe"]  # {label: (members_list, method)}
TTA_ACCEPTED = loaded["tta_enabled"]
for name, spec in MEMBERS.items():
    spec["dir"] = _find_dir("/kaggle/input", spec["kaggle_dataset"])
    print(name, "->", spec["dir"])
    assert spec["dir"] is not None and spec["dir"].exists(), f"Attach {spec['kaggle_dataset']}"

print(f"Recipe version {loaded['raw']['version']} ({loaded['raw']['created']}), "
      f"TTA_ACCEPTED={TTA_ACCEPTED}")
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available(),
      "device count:", torch.cuda.device_count() if torch.cuda.is_available() else 0)

In [ ]:
SLICES_PER_SERIES = 24
IMAGE_SIZE = 384
OUT_SIZE = nearest_patch_multiple(IMAGE_SIZE, _patch_size_for_backbone(next(iter(MEMBERS.values()))["backbone"]))  # 378 — same for all 3 members (all patch14)

GLOBAL_MEMBERS = list(MEMBERS.keys())

# fp16, not bf16 (2026-09-11 T4 adaptation): the pinned Kaggle accelerator is T4 x2 — T4 is sm_75
# (Turing), with no native bf16 tensor-core support; bf16 autocast on it either falls back to a
# slow/emulated path or errors depending on the torch build. The efficiency notebook already used
# fp16 for exactly this "smaller/older GPU" reason (cell-0 there); infer_core.run_study_inference[_tta]
# still DEFAULT to bf16 (correct for H100-based HPC rehearsal/dev, left untouched) — this notebook
# now passes fp16 explicitly at every call site instead of relying on that default.
AUTOCAST_DTYPE = torch.float16

# Decode overlap (2026-09-11): studies are decoded in a background thread pool (infer_core.
# prefetch_decode) while the GPU works the previous study, instead of the old decode-study-N-then-
# infer-study-N lockstep that left the CPU idle during every inference and the GPU idle during
# every decode. Worker count follows the same "trust os.cpu_count() on Kaggle" reasoning as the
# efficiency notebook (that cell's comment has the full 192-vs-16 HPC-SLURM backstory).
DECODE_WORKERS = max(1, min((os.cpu_count() or 4) - 1, 32))

BUDGET_SECONDS = 8 * 3600  # 9h hard cap (CLAUDE.md §1) minus margin for scheduling/write
CALIBRATION_STUDIES = 10   # measure actual per-study cost on this many before picking a ladder rung

DEVICE = "cpu"
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    cap = major * 10 + minor
    supported_caps = [int(a[3:]) for a in torch.cuda.get_arch_list() if a.startswith("sm_")]
    if supported_caps and cap >= min(supported_caps):
        DEVICE = "cuda"
    else:
        print(f"  [WARN] GPU {torch.cuda.get_device_name(0)} (sm_{cap}) unsupported by this torch "
              f"build (min sm_{min(supported_caps) if supported_caps else '?'}) — CPU fallback.")

# Multi-GPU member sharding (2026-09-11, T4 x2): each member's 5 folds run on ONE assigned device,
# and different members' devices are driven from separate Python threads (cell-5) so their CUDA
# kernels actually overlap instead of queuing behind each other on a single device — PyTorch's C++
# backend releases the GIL during a CUDA call, so two threads targeting cuda:0/cuda:1 get genuine
# concurrent GPU execution, not just concurrent Python bookkeeping. Falls back to everything on one
# device (today's behavior) when only one GPU is visible (e.g. a normal single-GPU HPC job).
N_GPUS = torch.cuda.device_count() if DEVICE == "cuda" else 0


def _assign_member_devices(members: dict, n_gpus: int) -> dict:
    if n_gpus <= 1:
        return {name: DEVICE for name in members}
    loads = [0.0] * n_gpus
    assignment = {}
    # Heaviest (ViT-B, cost 3) first so the greedy balance doesn't strand a big model behind an
    # already-loaded GPU — same relative-cost weighting as relative_cost() below.
    for name, spec in sorted(members.items(),
                              key=lambda kv: -(3.0 if kv[1]["backbone"].startswith("vit_base") else 1.0)):
        cost = 3.0 if spec["backbone"].startswith("vit_base") else 1.0
        gpu = min(range(n_gpus), key=lambda i: loads[i])
        assignment[name] = f"cuda:{gpu}"
        loads[gpu] += cost
    return assignment


MEMBER_DEVICE = _assign_member_devices(MEMBERS, N_GPUS)
print("device:", DEVICE, "n_gpus:", N_GPUS, "out_size:", OUT_SIZE, "decode_workers:", DECODE_WORKERS,
      "TTA_ACCEPTED:", TTA_ACCEPTED)
print("member device assignment:", MEMBER_DEVICE)

In [ ]:
def load_member_models(spec: dict, device: str) -> list:
    # Weight datasets are staged FLAT (foldN_best.pt at the dataset root, no subdirectories,
    # 2026-09-11) specifically to avoid a documented Kaggle risk: uploading per-fold
    # SUBDIRECTORIES requires --dir-mode zip (plain upload silently DROPS subdirectories
    # entirely), and the 2026-09-08 Kaggle debugging arc (EXPERIMENTS.md) found that zip-based
    # subdirectory upload does not reliably preserve folder nesting on extraction — fine for the
    # code package (unique .py filenames, just find __init__.py wherever it lands) but every
    # fold's checkpoint is identically named "best.pt", so a flattening extraction could silently
    # collide 5 files into 1. Falls back to the foldN/best.pt subdirectory layout for robustness
    # in case a dataset is ever staged that way instead.
    ckpt_paths = sorted(spec["dir"].glob("fold*_best.pt"))
    if not ckpt_paths:
        ckpt_paths = [d / "best.pt" for d in sorted(spec["dir"].glob("fold*"))
                      if d.is_dir() and (d / "best.pt").exists()]
    models = []
    for ckpt_path in ckpt_paths:
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model = SlotModel(spec["backbone"], num_targets=len(TARGET_COLUMNS), pretrained=False).to(device)
        model.load_state_dict(ckpt["ema"])
        model.eval()
        models.append(model)
    return models

member_models, member_rank_refs = {}, {}
for name, spec in MEMBERS.items():
    member_device = MEMBER_DEVICE[name]
    member_models[name] = load_member_models(spec, member_device)
    npz_path = spec["dir"] / "rank_reference.npz"
    member_rank_refs[name] = dict(np.load(npz_path)) if npz_path.exists() else {}
    print(f"{name} -> {member_device}: {len(member_models[name])}/5 folds loaded, rank_reference "
          f"{'found' if npz_path.exists() else 'MISSING (rank-mean labels fall back to prob-mean)'}")

if all(len(m) == 0 for m in member_models.values()):
    raise RuntimeError("No member models loaded at all — cannot run inference.")

In [ ]:
test_df = pd.read_csv(TEST_DIR / "test.csv")
series_df = pd.read_csv(TEST_DIR / "test_series.csv").rename(
    columns={"StudyInstanceUID": "study_uid", "SeriesInstanceUID": "series_uid"}
)
series_root = TEST_DIR / "test_series"
print(f"{len(test_df)} test studies, {len(series_df)} series.")

In [ ]:
# Ladder rungs, cheapest-fallback last. Each rung = (use_tta, members_to_use). TTA is measured
# OOF-negative (frozen recipe file) so it never appears as an active rung, only smaller member sets.
LADDER = [
    ("no-TTA + 3 members", False, GLOBAL_MEMBERS),
    ("no-TTA + 2 members", False, ["exp006_dinov2b_slot_v3labels", "exp005_dinov2s_slot_v3labels"]),
    ("smallest model only", False, ["exp005_dinov2s_slot_v3labels"]),
]

# Members run in separate threads (one per member, cell-2's MEMBER_DEVICE says which GPU each
# lands on) so members on DIFFERENT devices actually execute concurrently — members sharing a
# device just serialize on that device's stream as before, no loss either way. max_workers=3 (not
# N_GPUS) since two members can share a device without harm; the thread pool itself is cheap.
_member_pool = ThreadPoolExecutor(max_workers=max(1, len(GLOBAL_MEMBERS)))


def relative_cost(use_tta: bool, members: list[str]) -> float:
    """Rough relative inference cost vs. one ViT-S forward pass — ViT-B counted as 3x ViT-S
    (roughly matches parameter-count scaling), TTA doubles whichever members run."""
    unit = sum(3.0 if MEMBERS[m]["backbone"].startswith("vit_base") else 1.0 for m in members)
    return unit * (2.0 if use_tta else 1.0)


def run_member(name: str, slots, tag_sides, use_tta: bool) -> np.ndarray:
    models = member_models[name]
    device = MEMBER_DEVICE[name]
    if not models:
        return prior_row()
    if use_tta:
        return run_study_inference_tta(slots, tag_sides, models, device, SLICES_PER_SERIES, OUT_SIZE, AUTOCAST_DTYPE)
    return run_study_inference(slots, tag_sides, models, device, SLICES_PER_SERIES, OUT_SIZE, AUTOCAST_DTYPE)


def run_ensemble(slots, tag_sides, use_tta: bool, members: list[str]) -> np.ndarray:
    # Submit every member's forward pass before blocking on any result — with members split across
    # cuda:0/cuda:1 (T4 x2) this is what actually gets both GPUs busy at once (2026-09-11 T4
    # adaptation) instead of finishing one member's device-to-device-synchronous call before
    # starting the next.
    futures = {m: _member_pool.submit(run_member, m, slots, tag_sides, use_tta) for m in members}
    member_probs = {m: f.result() for m, f in futures.items()}
    # RECIPE (from the frozen recipe file) already gives {label: (members, method)} for the full
    # 3-member set; when the ladder has dropped to fewer members, use that label's method but
    # restrict to whichever of those members are actually active at this rung.
    active_recipe = {label: ([m for m in orig_members if m in members], method)
                      for label, (orig_members, method) in RECIPE.items()}
    return ensemble_blend(member_probs, active_recipe, member_rank_refs)

In [ ]:
import gc

guard = WallClockGuard(BUDGET_SECONDS)


def _study_task(study_uid: str) -> tuple:
    study_series = series_df[series_df["study_uid"] == study_uid].to_dict("records")
    return (study_uid, study_series, str(series_root), SLICES_PER_SERIES, OUT_SIZE)


# --- Calibration: run the full (top) rung on the first few studies through the SAME decode-
# overlap pipeline as the main loop (prefetch_decode — CPU decode of study N+1 runs in a
# background thread pool while the GPU is still working study N, 2026-09-11 T4 adaptation), so
# the measured per-study cost reflects the actual overlapped pipeline, not a serial one. Project
# total time for every rung, pick the highest one that fits with margin — one rung is used for the
# WHOLE run (not switched mid-way) so every study is scored consistently.
n_calib = min(CALIBRATION_STUDIES, len(test_df))
calib_uids = list(test_df["StudyInstanceUID"].iloc[:n_calib])
calib_tasks = [_study_task(u) for u in calib_uids]

calib_t0 = time.time()
calib_rows = []
for study_uid, slots, tag_sides, err in prefetch_decode(calib_tasks, decode_one_for_pool, DECODE_WORKERS):
    if slots and err is None:
        # inference_mode also applied via infer_core._run_orientation's own decorator
        # (thread-local, so it must live on the actual forward-pass function to cover
        # _member_pool's worker threads) -- wrapped here too for an explicit, auditable
        # no-autograd-state guarantee at every call site (2026-09-16 OOM fix).
        with torch.inference_mode():
            probs = run_ensemble(slots, tag_sides, *LADDER[0][1:])
        calib_rows.append({"StudyInstanceUID": study_uid, **{c: float(p) for c, p in zip(TARGET_COLUMNS, probs)}})
        del slots
calib_elapsed = time.time() - calib_t0
per_study_at_rung0 = calib_elapsed / max(n_calib, 1)
rung0_cost = relative_cost(*LADDER[0][1:])
print(f"Calibration: {n_calib} studies, {calib_elapsed:.1f}s ({per_study_at_rung0:.2f}s/study at rung 0, decode-overlapped)")

chosen_rung_idx = 0
for idx, (label, use_tta, members) in enumerate(LADDER):
    projected_per_study = per_study_at_rung0 * (relative_cost(use_tta, members) / rung0_cost)
    projected_total = projected_per_study * len(test_df)
    fits = projected_total <= guard.budget * 0.85  # 15% margin under budget
    print(f"  rung {idx} ({label}): projected {projected_total / 60:.1f} min for {len(test_df)} studies "
          f"-> {'FITS' if fits else 'too slow'}")
    if fits:
        chosen_rung_idx = idx
        break
else:
    chosen_rung_idx = len(LADDER) - 1  # nothing fit comfortably — use the cheapest rung anyway

rung_label, USE_TTA, ACTIVE_MEMBERS = LADDER[chosen_rung_idx]
print(f"Chosen rung: {chosen_rung_idx} ({rung_label})")

rows = list(calib_rows) if chosen_rung_idx == 0 else []  # reuse calibration results only if they match the chosen rung
done_uids = {r["StudyInstanceUID"] for r in rows}
remaining_uids = [u for u in test_df["StudyInstanceUID"] if u not in done_uids]
remaining_tasks = [_study_task(u) for u in remaining_uids]

n_prior_fallback = n_dual_flip = 0
processed = len(rows)
# Guard is checked at the TOP of each iteration, before touching the item prefetch_decode just
# handed back — if it trips, we stop consuming the generator and bulk-fill everything left with
# priors in one shot (cheaper than looping through each remaining study individually). The
# already-decoded item that triggered the check is simply discarded. Note: prefetch_decode's
# background feeder thread (daemon=True) may still be mid-decode on a few more studies at the
# moment we stop consuming — harmless (wasted CPU, no correctness impact), dies with the kernel;
# not worth generator-cancellation machinery for a rare, graceful-degradation path in a one-shot
# submission notebook.
for study_uid, slots, tag_sides, err in prefetch_decode(remaining_tasks, decode_one_for_pool, DECODE_WORKERS):
    if guard.over(margin=0.05):
        break
    row = {"StudyInstanceUID": study_uid}
    try:
        if err is not None or not slots:
            raise RuntimeError(err or "no series decoded successfully for this study")
        if all(s not in ("R", "L") for s in tag_sides):
            n_dual_flip += 1
        with torch.inference_mode():
            probs = run_ensemble(slots, tag_sides, USE_TTA, ACTIVE_MEMBERS)
        row.update({c: float(p) for c, p in zip(TARGET_COLUMNS, probs)})
    except Exception as e:
        print(f"  [WARN] {study_uid}: {type(e).__name__}: {e} — using label priors")
        n_prior_fallback += 1
        row.update({c: float(p) for c, p in zip(TARGET_COLUMNS, prior_row())})
    del slots  # decoded volumes are the biggest per-study allocation (up to 6 slots x
    # T x out_size x out_size float32 each) -- drop the reference as soon as this
    # study's row is built, don't wait for the next loop iteration's reassignment
    rows.append(row)
    processed += 1
    if processed % 25 == 0:
        gc.collect()  # 2026-09-16 Kaggle OOM fix: reclaim decoded-volume memory on a
        # fixed cadence rather than trusting refcounting + the allocator alone over a
        # ~1300-study run
    if processed % 100 == 0 or processed == len(test_df):
        print(f"  {processed}/{len(test_df)} studies, {guard.elapsed():.0f}s elapsed ({guard.fraction_used():.1%} of budget)")

if processed < len(test_df):
    done_now = {r["StudyInstanceUID"] for r in rows}
    skipped = [u for u in remaining_uids if u not in done_now]
    print(f"  [BUDGET] {guard.elapsed():.0f}s used, {processed}/{len(test_df)} done — priors for the rest.")
    for uid in skipped:
        rows.append({"StudyInstanceUID": uid, **{c: float(p) for c, p in zip(TARGET_COLUMNS, prior_row())}})
        n_prior_fallback += 1

print(f"Done: {len(rows)} studies, rung={chosen_rung_idx} ({rung_label}), {n_prior_fallback} full-prior fallbacks, "
      f"{n_dual_flip} dual-flip, {guard.elapsed():.0f}s total.")

In [ ]:
sub_df = pd.DataFrame(rows)[["StudyInstanceUID"] + TARGET_COLUMNS]

assert len(sub_df) == len(test_df), f"row count mismatch: {len(sub_df)} vs {len(test_df)}"
assert set(sub_df["StudyInstanceUID"]) == set(test_df["StudyInstanceUID"]), "study UID mismatch"
assert not sub_df[TARGET_COLUMNS].isna().any().any(), "NaNs in submission"
assert (sub_df[TARGET_COLUMNS] >= 0).all().all() and (sub_df[TARGET_COLUMNS] <= 1).all().all(), "probabilities out of [0,1]"
assert list(sub_df.columns) == ["StudyInstanceUID"] + TARGET_COLUMNS, f"header mismatch: {list(sub_df.columns)}"

sub_df.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}: {len(sub_df)} rows, all checks passed. Rung used: {chosen_rung_idx} ({rung_label}).")
sub_df.head()